# WSN Lab 1: Multi-Sensor Edge Node Data Analysis
This notebook analyses the 600-row combined dataset. Upload the CSV when prompted.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from pathlib import Path
import io

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print('Dataset shape:', df.shape)
display(df.head())

In [ ]:
numeric_columns = ['timestamp','temp','humidity','light','motion','temp_cal','temp_filt','delta_temp','valid','dht_fresh','event','ref_temp','t_s']
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)
print('Missing values:')
display(df.isna().sum().to_frame('missing_values'))
print('Valid DHT rows:', int(df['valid'].fillna(0).eq(1).sum()))
print('Invalid DHT rows:', int(df['valid'].fillna(0).eq(0).sum()))
print('Motion rows:', int(df['motion'].fillna(0).eq(1).sum()))
print('Event transitions:', int(df['event'].fillna(0).eq(1).sum()))

In [ ]:
measurements = ['temp','humidity','light','temp_cal','temp_filt','delta_temp','ref_temp']
stats = df[[c for c in measurements if c in df.columns]].describe().T
display(stats)
stats.to_csv('summary_statistics.csv')

## 1. Temperature readings

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['timestamp'], df['temp'], label='DHT temperature')
if 'ref_temp' in df.columns:
    plt.plot(df['timestamp'], df['ref_temp'], label='LM35 reference', alpha=0.8)
plt.xlabel('ESP32 timestamp (seconds since boot)')
plt.ylabel('Temperature (°C)')
plt.title('Temperature Readings Over Time')
plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()

## 2. Humidity readings

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['timestamp'], df['humidity'])
plt.xlabel('ESP32 timestamp (seconds since boot)')
plt.ylabel('Relative humidity (%)')
plt.title('Humidity Readings Over Time')
plt.grid(True); plt.tight_layout(); plt.show()

## 3. Calibrated and filtered temperature

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['timestamp'], df['temp_cal'], label='Calibrated temperature', alpha=0.65)
plt.plot(df['timestamp'], df['temp_filt'], label='Filtered temperature', linewidth=2)
plt.xlabel('ESP32 timestamp (seconds since boot)')
plt.ylabel('Temperature (°C)')
plt.title('Effect of Moving-Average Filtering')
plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()

## 4. LDR light readings
The LDR values are normalised readings from 0 to 1, not lux.

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['timestamp'], df['light'])
plt.xlabel('ESP32 timestamp (seconds since boot)')
plt.ylabel('Normalised LDR reading (0 to 1)')
plt.title('Light Sensor Response Over Time')
plt.ylim(0,1.05); plt.grid(True); plt.tight_layout(); plt.show()

## 5. PIR motion detection

In [ ]:
plt.figure(figsize=(10,3.5))
plt.step(df['timestamp'], df['motion'], where='post')
plt.xlabel('ESP32 timestamp (seconds since boot)')
plt.ylabel('PIR state (0 = no motion, 1 = motion)')
plt.title('PIR Motion Detection')
plt.yticks([0,1], ['No motion','Motion']); plt.ylim(-0.15,1.15)
plt.grid(True); plt.tight_layout(); plt.show()

## 6. Event transitions

In [ ]:
events = df[df['event'].eq(1)]
plt.figure(figsize=(10,4))
plt.scatter(events['timestamp'], events['event'])
plt.xlabel('ESP32 timestamp (seconds since boot)')
plt.ylabel('Event')
plt.title('Detected Event Transitions')
plt.yticks([1], ['Event']); plt.ylim(0.5,1.5)
plt.grid(True); plt.tight_layout(); plt.show()

## Export the analysis outputs

In [ ]:
from pathlib import Path
out = Path('results')
figdir = out / 'figures'
figdir.mkdir(parents=True, exist_ok=True)
stats.to_csv(out / 'summary_statistics.csv')
quality = df.isna().sum().to_string()
(out / 'data_quality_summary.txt').write_text(quality, encoding='utf-8')
print('Summary files saved in results/. Save/download your figures from the cells above if needed.')